# Manual curation in the SpikeInterface GUI (Spyglass pipeline, step 3 of 4)

This notebook hand-curates the sorting produced by the Spyglass pipeline using the
[SpikeInterface GUI](https://github.com/SpikeInterface/spikeinterface-gui). It is the one step in
the chain that does **not** run in the `spyglass` environment.

> **Environment:** run this notebook with the **`spikeinterface_gui_env`** kernel
> (SpikeInterface 0.104 + `spikeinterface-gui`). The `spyglass` environment has SpikeInterface 0.99
> and no GUI, so it cannot open the analyzer or launch the GUI.

**Inputs:** the `recording` and `sorting` folders exported by
[`Pipeline_Spyglass_Curation.ipynb`](Pipeline_Spyglass_Curation.ipynb).
**Output:** a `curation_data.json` file that
[`Pipeline_Spyglass_CompareCurations.ipynb`](Pipeline_Spyglass_CompareCurations.ipynb) re-ingests
back into Spyglass.

In [ ]:
from pathlib import Path

import spikeinterface as si
from spikeinterface_gui import run_mainwindow

## Point at the exported sorting

Set `export_dir` to the path printed by `Pipeline_Spyglass_Curation.ipynb` (it ends in the
`sorting_id`).

In [ ]:
# Paste the sorting_id printed by Pipeline_Spyglass_Curation.ipynb:
sorting_id = "PASTE_SORTING_ID_HERE"
export_dir = Path("manual_curation_export") / sorting_id

assert (export_dir / "recording").exists() and (export_dir / "sorting").exists(), (
    f"Exported recording/sorting not found under {export_dir}. "
    "Run Pipeline_Spyglass_Curation.ipynb (step 2) first and copy the path it prints."
)

## Load the recording and sorting

These were written by SpikeInterface 0.99 in the `spyglass` environment; here we read them with
SpikeInterface 0.104. The npz sorting and binary recording formats are portable across these
versions (you may see a version-mismatch warning, which is safe to ignore). If loading fails, see
the README for the NWB-extractor fallback.

In [ ]:
recording = si.load(export_dir / "recording")
sorting = si.load(export_dir / "sorting")
print(recording)
print(sorting)

## Build a SortingAnalyzer

The GUI is driven by a `SortingAnalyzer` plus a set of computed extensions (waveforms, templates,
amplitudes, correlograms, quality metrics, ...). We build it once and save it next to the export so
the GUI can write its curation file alongside.

> ⏱️ Computing the extensions takes a few minutes.

In [ ]:
analyzer = si.create_sorting_analyzer(
    sorting,
    recording,
    folder=export_dir / "sorting_analyzer",
    format="binary_folder",
    overwrite=True,
)
analyzer.compute(
    [
        "random_spikes",
        "waveforms",
        "templates",
        "noise_levels",
        "spike_amplitudes",
        "correlograms",
        "unit_locations",
        "template_similarity",
    ]
)
analyzer.compute("quality_metrics")

## Launch the GUI and curate

Running the next cell opens the desktop GUI with curation enabled. In the GUI you can:

- **merge** units that were over-split (select them, then merge),
- **label** units (`good` / `noise` / `MUA`), and
- **remove** units that are noise.

When you are done, **save** from within the GUI (it writes
`sorting_analyzer/spikeinterface_gui/curation_data.json` inside `export_dir`). Then move on to
`Pipeline_Spyglass_CompareCurations.ipynb`.

> **Terminal alternative.** Instead of this cell you can launch the same GUI from a shell:
> ```
> conda run -n spikeinterface_gui_env sigui <export_dir>/sorting_analyzer --curation --mode desktop
> ```

In [ ]:
run_mainwindow(analyzer, mode="desktop", curation=True)

## Done

The GUI saved your curation to:

```
<export_dir>/sorting_analyzer/spikeinterface_gui/curation_data.json
```

Switch back to the **`spyglass`** kernel and run
[`Pipeline_Spyglass_CompareCurations.ipynb`](Pipeline_Spyglass_CompareCurations.ipynb) to ingest it
and compare against the raw and automatic curations.